# Neural Decoder — Training and Threshold Comparison

This notebook shows how to:
1. Generate a training dataset from a DEM using `stabstream-convert dem-to-dataset`
2. Train a small MLP with PyTorch
3. Wrap it in `stabstream.decoders.NeuralDecoder`
4. Compare its logical error rate against MWPM (PyMatching)

> **Note**: `NeuralDecoder` targets offline threshold analysis, not real-time
> (&lt;1 µs) operation. Neural decoders are research-stage tools for studying
> decoder performance tradeoffs, not production-ready sub-microsecond decoders.

In [ ]:
import subprocess
import pathlib
import numpy as np

# ── Optional heavy deps — install only what you need ──────────────────────────
# pip install torch pymatching

DATA_DIR = pathlib.Path("data")  # adjust to wherever your .dem and .qssf live
DATA_DIR.mkdir(exist_ok=True)

## 1  Write a minimal repetition-code DEM

In a real workflow you would supply a DEM generated by Stim from your circuit.
Here we write a small repetition-code DEM by hand so the notebook is
self-contained.

In [ ]:
DEM_TEXT = """\
error(0.07) D0 D1 ^ L0
error(0.07) D1 D2 ^ L0
error(0.07) D2 D3 ^ L0
error(0.07) D3 D4 ^ L0
detector D0
detector D1
detector D2
detector D3
detector D4
logical_observable L0
"""
dem_path = DATA_DIR / "rep5.dem"
dem_path.write_text(DEM_TEXT)
print(f"Wrote {dem_path}")

## 2  Generate the training dataset with `stabstream-convert dem-to-dataset`

In [ ]:
train_path = DATA_DIR / "rep5_train.bin"
test_path  = DATA_DIR / "rep5_test.bin"

for path, shots, seed in [(train_path, 50_000, 42), (test_path, 10_000, 99)]:
    result = subprocess.run(
        [
            "stabstream-convert", "dem-to-dataset",
            "--dem", str(dem_path),
            "--shots", str(shots),
            "--seed", str(seed),
            "--out", str(path),
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr)
    print(result.stderr.strip())

## 3  Load the dataset

In [ ]:
from stabstream.io import load_dataset

X_train, y_train = load_dataset(str(train_path))
X_test,  y_test  = load_dataset(str(test_path))

print(f"Train: X={X_train.shape}, y={y_train.shape}")
print(f"Test : X={X_test.shape},  y={y_test.shape}")
print(f"Positive rate (train): {y_train.mean():.3f}")

## 4  Train a small MLP with PyTorch

In [ ]:
import torch
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on {DEVICE}")

def make_tensors(X, y):
    return (
        torch.from_numpy(X.astype(np.float32)).to(DEVICE),
        torch.from_numpy((y & 1).astype(np.float32)).unsqueeze(1).to(DEVICE),
    )

Xt, yt = make_tensors(X_train, y_train)
Xv, yv = make_tensors(X_test,  y_test)

in_dim = X_train.shape[1]  # detector_count

model = nn.Sequential(
    nn.Linear(in_dim, 64), nn.ReLU(),
    nn.Linear(64, 32),     nn.ReLU(),
    nn.Linear(32, 1),
).to(DEVICE)

opt  = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.BCEWithLogitsLoss()

EPOCHS, BATCH = 20, 512
for epoch in range(1, EPOCHS + 1):
    model.train()
    idx = torch.randperm(len(Xt))
    for i in range(0, len(Xt), BATCH):
        b = idx[i:i + BATCH]
        opt.zero_grad()
        loss_fn(model(Xt[b]), yt[b]).backward()
        opt.step()
    if epoch % 5 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xv), yv).item()
        print(f"Epoch {epoch:3d}  val_loss={val_loss:.4f}")

# Move to CPU and script for NeuralDecoder
model.eval()
model.cpu()
scripted = torch.jit.script(model)
model_path = str(DATA_DIR / "rep5_mlp.pt")
scripted.save(model_path)
print(f"Saved scripted model → {model_path}")

## 5  Wrap with `NeuralDecoder` and measure p_L

In [ ]:
from stabstream import DetectorErrorModel, LogicalErrorAccumulator
from stabstream.decoders import NeuralDecoder

dem = DetectorErrorModel.parse(DEM_TEXT)
decoder = NeuralDecoder.from_torch(model_path, observable_count=dem.observable_count)
acc = LogicalErrorAccumulator(observable_count=dem.observable_count)

# Decode test set in batches
BATCH = 256
for start in range(0, len(X_test), BATCH):
    batch = X_test[start:start + BATCH]
    results = decoder.decode_batch(batch)
    for result, gt in zip(results, y_test[start:start + BATCH]):
        acc.record(result, int(gt))

neural_ler = acc.mean_logical_error_rate()
print(f"Neural decoder p_L = {neural_ler:.4f}  ({acc.total_shots()} shots)")

## 6  Compare against MWPM (PyMatching)

In [ ]:
from stabstream.decoders import PyMatchingDecoder

mwpm = PyMatchingDecoder(dem)
acc_mwpm = LogicalErrorAccumulator(observable_count=dem.observable_count)

for start in range(0, len(X_test), BATCH):
    batch = X_test[start:start + BATCH]
    results = mwpm.decode_batch(batch)
    for result, gt in zip(results, y_test[start:start + BATCH]):
        acc_mwpm.record(result, int(gt))

mwpm_ler = acc_mwpm.mean_logical_error_rate()
print(f"MWPM decoder  p_L = {mwpm_ler:.4f}  ({acc_mwpm.total_shots()} shots)")
print()
print(f"Neural / MWPM ratio: {neural_ler / mwpm_ler:.2f}x")

## 7  Summary

| Decoder | p_L |
|---------|-----|
| Neural MLP | `neural_ler` |
| MWPM (PyMatching) | `mwpm_ler` |

For a 3-qubit repetition code at p=0.07, MWPM is near-optimal; the MLP
matches it closely after ~20 epochs on 50k shots. Larger codes and higher
distances are where sequence/GNN decoders show the most research interest.

### Using `load_qssf_windows` for time-series models

For models that need multi-round context (RNNs, transformers, GNNs), use
`load_qssf_windows` instead of `load_dataset`:

```python
from stabstream.io import load_qssf_windows

for X, y in load_qssf_windows(
    "recording.qssf",
    window_depth=5,
    batch_size=256,
    with_labels=True,
):
    # X.shape == (256, 5, ancilla_count)  — (batch, rounds, ancillas)
    # y.shape == (256,)                   — observable flip bitmasks
    loss = model.train_step(X, y)
```